# LeetCode 3276 — Select Cells in Grid With Maximum Score bằng Python

- Link: https://leetcode.com/problems/select-cells-in-grid-with-maximum-score/
- Độ khó: **Hard**
- Chủ đề: **Đổi vai**: duyệt theo **giá trị** (≤ 100) thay vì theo ô, giữ **bitmask các hàng đã dùng** (≤ 10 hàng) ⇒ DP `O(V · 2^R · R)`; góc nhìn **ghép cặp trọng số lớn nhất** (hàng ↔ giá trị)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxScore(self, grid: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Hai ràng buộc: mỗi **hàng** chọn tối đa một ô; các **giá trị** chọn đôi một khác nhau ⇒ mỗi giá trị dùng tối đa một lần, mỗi hàng tối đa một lần — một bài **ghép cặp** hàng ↔ giá trị.
2. Kích thước nhỏ: `≤ 10` hàng ⇒ bitmask hàng; giá trị `≤ 100` ⇒ duyệt theo giá trị.
3. DP: `dp[mask]` = tổng lớn nhất khi các hàng trong `mask` đã được dùng, sau khi xét một số giá trị; với giá trị `v`, hoặc bỏ qua, hoặc gán cho một hàng `r ∉ mask` có chứa `v`.
4. Cách khác: luồng chi phí nhỏ nhất / ghép cặp trọng số lớn nhất.
5. Kiểm chứng bằng quay lui.

## 2. Đề bài & ví dụ

Cho ma trận `grid` gồm số nguyên dương. Chọn **một hoặc nhiều** ô sao cho: không có hai ô cùng hàng; các giá trị được chọn đôi một khác nhau. Điểm là tổng các giá trị được chọn. Trả về điểm lớn nhất.

**Ví dụ 1:** `[[1,2,3],[4,3,2],[1,1,1]]` → `8` (`3, 4, 1`).
**Ví dụ 2:** `[[8,7,6],[8,3,2]]` → `15` (`7, 8`).

**Ràng buộc:** `1 ≤ số hàng, số cột ≤ 10`; `1 ≤ grid[i][j] ≤ 100`.

In [1]:
from typing import List
from collections import defaultdict
from functools import lru_cache

examples = [([[1, 2, 3], [4, 3, 2], [1, 1, 1]], 8), ([[8, 7, 6], [8, 3, 2]], 15)]
for g, exp in examples:
    print(g, '-> kỳ vọng', exp)

[[1, 2, 3], [4, 3, 2], [1, 1, 1]] -> kỳ vọng 8
[[8, 7, 6], [8, 3, 2]] -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. Bảng "giá trị → tập hàng chứa nó"

Mỗi giá trị `v` chỉ quan tâm **hàng nào** chứa nó (vị trí cột không quan trọng).

In [2]:
def value_rows(grid):
    rows = defaultdict(set)
    for r, row in enumerate(grid):
        for v in row:
            rows[v].add(r)
    return rows

print(dict(value_rows(examples[0][0])))

{1: {0, 2}, 2: {0, 1}, 3: {0, 1}, 4: {1}}


### 3.2. Vì sao không duyệt theo hàng?

Nếu DP theo hàng, ta phải nhớ **tập giá trị đã dùng** — tới `2^100` trạng thái. Ngược lại, DP theo giá trị chỉ cần nhớ **tập hàng đã dùng** — `2^10 = 1024` trạng thái. Đổi vai chiều nhỏ làm trạng thái là kỹ thuật then chốt.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 `[[8,7,6],[8,3,2]]`: giá trị `8` có ở cả hai hàng — chỉ được dùng một lần. Nếu hàng 0 lấy `8` thì hàng 1 lấy tốt nhất `3` ⇒ `11`; nếu hàng 1 lấy `8` thì hàng 0 lấy `7` ⇒ `15`. Tham lam "mỗi hàng lấy max" (`8, 8`) vi phạm ràng buộc giá trị.

## 5. Cách 1 — Brute Force: quay lui theo hàng (chọn một ô hoặc bỏ)

In [3]:
def brute(grid):
    R = len(grid)
    best = 0
    def go(r, used, total):
        nonlocal best
        if r == R:
            best = max(best, total)
            return
        go(r + 1, used, total)                       # bỏ hàng r
        for v in set(grid[r]):
            if v not in used:
                used.add(v); go(r + 1, used, total + v); used.remove(v)
    go(0, set(), 0)
    return best

for g, exp in examples:
    print(brute(g), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15


**Độ phức tạp:** `O((C + 1)^R)` — `11^10` trong trường hợp xấu, chỉ cho lưới nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
dp[mask] = −∞, dp[0] = 0
for v in các giá trị phân biệt:
    new = dp[:]                                  # bỏ qua v
    for mask: for r in rows[v], r ∉ mask:
        new[mask | 1<<r] = max(new[...], dp[mask] + v)
    dp = new
return max(dp)
```

`O(V · 2^R · R) ≤ 100 · 1024 · 10 ≈ 10^6`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ `(chỉ số giá trị, mask)`

In [4]:
def memo_solution(grid):
    rows = value_rows(grid)
    vals = sorted(rows, reverse=True)

    @lru_cache(maxsize=None)
    def f(i, mask):
        if i == len(vals):
            return 0
        v = vals[i]
        best = f(i + 1, mask)
        for r in rows[v]:
            if not mask >> r & 1:
                best = max(best, v + f(i + 1, mask | 1 << r))
        return best

    ans = f(0, 0)
    f.cache_clear()
    return ans

for g, exp in examples:
    print(memo_solution(g), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15


### 7.2. Cách 3 — DP lặp theo giá trị với bitmask hàng (lời giải chính)

In [5]:
def bitmask_dp(grid):
    R = len(grid)
    rows = value_rows(grid)
    NEG = -1
    dp = [NEG] * (1 << R)
    dp[0] = 0
    for v, rs in rows.items():
        new = dp[:]
        for mask in range(1 << R):
            base = dp[mask]
            if base < 0:
                continue
            for r in rs:
                if not mask >> r & 1:
                    m2 = mask | 1 << r
                    if base + v > new[m2]:
                        new[m2] = base + v
        dp = new
    return max(dp)

for g, exp in examples:
    print(bitmask_dp(g), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15


Thứ tự duyệt giá trị không quan trọng (mỗi giá trị được xét đúng một lần), miễn dùng bản sao `new` để không gán một giá trị cho hai hàng.

### 7.3. Cách 4 — Ghép cặp trọng số lớn nhất bằng luồng chi phí nhỏ nhất

Đồ thị: nguồn → mỗi hàng (sức chứa 1), hàng → giá trị nó chứa (sức chứa 1, chi phí `−v`), giá trị → đích (sức chứa 1). Tăng luồng theo đường chi phí âm nhỏ nhất (Bellman–Ford) tới khi không còn đường chi phí âm. Đây là cách tổng quát khi số hàng lớn (không bitmask được).

In [6]:
def min_cost_flow_solution(grid):
    R = len(grid)
    vals = sorted(value_rows(grid))
    vid = {v: R + 1 + i for i, v in enumerate(vals)}
    S, T = 0, R + 1 + len(vals)
    N = T + 1
    graph = [[] for _ in range(N)]                    # cạnh: [đích, sức chứa, chi phí, chỉ số cạnh ngược]
    def add(u, w, cap, cost):
        graph[u].append([w, cap, cost, len(graph[w])])
        graph[w].append([u, 0, -cost, len(graph[u]) - 1])
    for r in range(R):
        add(S, r + 1, 1, 0)
        for v in set(grid[r]):
            add(r + 1, vid[v], 1, -v)
    for v in vals:
        add(vid[v], T, 1, 0)
    total = 0
    while True:
        dist = [float('inf')] * N; dist[S] = 0; prev = [None] * N
        for _ in range(N):                             # Bellman–Ford
            changed = False
            for u in range(N):
                if dist[u] == float('inf'):
                    continue
                for idx, (w, cap, cost, _) in enumerate(graph[u]):
                    if cap > 0 and dist[u] + cost < dist[w]:
                        dist[w] = dist[u] + cost; prev[w] = (u, idx); changed = True
            if not changed:
                break
        if dist[T] >= 0:                               # không còn đường làm tăng tổng
            break
        w = T
        while w != S:
            u, idx = prev[w]
            graph[u][idx][1] -= 1
            back = graph[u][idx][3]
            graph[w][back][1] += 1
            w = u
        total += -dist[T]
    return total

for g, exp in examples:
    print(min_cost_flow_solution(g), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15


Dừng khi đường ngắn nhất có chi phí `≥ 0`: thêm luồng nữa không làm tổng tăng (luồng chi phí nhỏ nhất theo từng đơn vị là lồi).

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1, in các `mask` có giá trị sau mỗi giá trị `v` (duyệt giảm dần).

In [7]:
def trace(grid):
    R = len(grid); rows = value_rows(grid)
    dp = {0: 0}
    for v in sorted(rows, reverse=True):
        new = dict(dp)
        for mask, base in dp.items():
            for r in rows[v]:
                if not mask >> r & 1:
                    m2 = mask | 1 << r
                    new[m2] = max(new.get(m2, -1), base + v)
        dp = new
        print(f"v = {v} (hàng {sorted(rows[v])}):", {format(m, f'0{R}b'): s for m, s in sorted(dp.items())})
    print('đáp án:', max(dp.values()), '| quay lui:', brute(grid))

trace(examples[0][0])

v = 4 (hàng [1]): {'000': 0, '010': 4}
v = 3 (hàng [0, 1]): {'000': 0, '001': 3, '010': 4, '011': 7}
v = 2 (hàng [0, 1]): {'000': 0, '001': 3, '010': 4, '011': 7}
v = 1 (hàng [0, 2]): {'000': 0, '001': 3, '010': 4, '011': 7, '100': 1, '101': 4, '110': 5, '111': 8}
đáp án: 8 | quay lui: 8


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def maxScore(self, grid: List[List[int]]) -> int:
        R = len(grid)
        rows = {}
        for r, row in enumerate(grid):
            for v in row:
                rows.setdefault(v, set()).add(r)
        full = 1 << R
        dp = [-1] * full
        dp[0] = 0
        for v, rs in rows.items():
            new = dp[:]
            bits = [1 << r for r in rs]
            for mask in range(full):
                base = dp[mask]
                if base < 0:
                    continue
                cand = base + v
                for b in bits:
                    if not mask & b and cand > new[mask | b]:
                        new[mask | b] = cand
            dp = new
        return max(dp)

for g, exp in examples:
    got = Solution().maxScore(g)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
15 OK


## 10. Vì sao thuật toán đúng

1. **Mô hình ghép cặp:** một lựa chọn hợp lệ ⇔ một tập cặp `(hàng r, giá trị v)` với `v` xuất hiện ở hàng `r`, mỗi hàng và mỗi giá trị trong tối đa một cặp. Điểm = tổng các `v`.
2. **DP theo giá trị:** sau khi xét tập giá trị `V'`, `dp[mask]` = tổng lớn nhất của một ghép cặp chỉ dùng giá trị trong `V'` và đúng các hàng `mask`. Xét giá trị mới `v`: hoặc không dùng (giữ nguyên), hoặc ghép với một hàng `r ∉ mask` chứa `v` — đủ mọi khả năng, không trùng lặp sai.
3. **Đáp án:** `max_mask dp[mask]` sau khi xét mọi giá trị. Mọi giá trị dương nên luôn chọn được ít nhất một ô.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | `O((C + 1)^R)` | `O(R)` |
| 2. Đệ quy có nhớ | `O(V · 2^R · R)` | `O(V · 2^R)` |
| 3. DP lặp | `O(V · 2^R · R)` | `O(2^R)` |
| 4. Luồng chi phí nhỏ nhất | `O(R · N · E)` (Bellman–Ford mỗi lần tăng) | `O(E)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'memo': memo_solution, 'bitmask_dp': bitmask_dp, 'min_cost_flow': min_cost_flow_solution,
         'submit': lambda g: Solution().maxScore(g)}
edge = [([[5]], 5), ([[1, 1, 1]], 1), ([[7], [7], [7]], 7), ([[1, 2], [1, 2]], 3), ([[100] * 10] * 10, 100),
        ([[3, 2], [3, 1], [2, 1]], 6)]
for g, exp in examples + edge:
    assert brute(g) == exp, (g, brute(g))
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với quay lui

In [10]:
import random
rng = random.Random(3276)
cnt = 0
for _ in range(300):
    R, C = rng.randint(1, 5), rng.randint(1, 4)
    V = rng.choice([3, 6, 20])
    g = [[rng.randint(1, V) for _ in range(C)] for _ in range(R)]
    exp = brute(g)
    for name, f in impls.items():
        assert f(g) == exp, (name, g)
    cnt += 1
print('Khớp quay lui trên', cnt, 'lưới ngẫu nhiên.')

Khớp quay lui trên 300 lưới ngẫu nhiên.


### 12.3. Ca lớn `10 × 10`

In [11]:
import time
big = {
    'ngẫu nhiên 1..100': [[rng.randint(1, 100) for _ in range(10)] for _ in range(10)],
    'giá trị 1..12 (xung đột nhiều)': [[rng.randint(1, 12) for _ in range(10)] for _ in range(10)],
    'mỗi hàng 91..100': [[91 + j for j in range(10)] for _ in range(10)],
}
for name, g in big.items():
    t = time.time(); a = Solution().maxScore(g); t1 = time.time() - t
    t = time.time(); b = min_cost_flow_solution(g); t2 = time.time() - t
    t = time.time(); c = memo_solution(g); t3 = time.time() - t
    assert a == b == c
    print(f"{name:32s}: {a:4d} | nộp {t1:.2f}s | luồng {t2:.2f}s | đệ quy {t3:.2f}s")
assert Solution().maxScore(big['mỗi hàng 91..100']) == sum(range(91, 101))
print('mỗi hàng 91..100: đáp án = 91 + ... + 100 = 955 — đúng.')

ngẫu nhiên 1..100               :  907 | nộp 0.01s | luồng 0.00s | đệ quy 0.04s
giá trị 1..12 (xung đột nhiều)  :   75 | nộp 0.00s | luồng 0.00s | đệ quy 0.01s
mỗi hàng 91..100                :  955 | nộp 0.00s | luồng 0.00s | đệ quy 0.01s
mỗi hàng 91..100: đáp án = 91 + ... + 100 = 955 — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: mỗi hàng lấy giá trị lớn nhất

In [12]:
def wrong_row_max(grid):
    return sum(max(row) for row in grid)                      # SAI: trùng giá trị

print('sai :', wrong_row_max(examples[1][0]), '| đúng:', Solution().maxScore(examples[1][0]))

sai : 16 | đúng: 15


**Vì sao sai:** `8` xuất hiện ở cả hai hàng; chỉ được tính một lần.

### Lỗi 2 — Tham lam: lấy giá trị lớn nhất toàn cục, gán cho hàng đầu tiên còn trống

In [13]:
def wrong_global_greedy(grid):
    rows = value_rows(grid); used = set(); total = 0
    for v in sorted(rows, reverse=True):
        free = sorted(r for r in rows[v] if r not in used)
        if free:
            used.add(free[0]); total += v                     # SAI: chọn hàng tùy tiện
    return total

bad = [[3, 2], [3, 1]]
print('sai :', wrong_global_greedy(bad), '| đúng:', Solution().maxScore(bad))

sai : 4 | đúng: 5


**Vì sao sai:** gán `3` cho hàng 0 thì hàng 1 chỉ còn `1` (tổng 4); gán `3` cho hàng 1 thì hàng 0 lấy `2` (tổng 5). Chọn hàng nào cho một giá trị ảnh hưởng tới các giá trị sau — cần DP/ghép cặp.

### Lỗi 3 — Quên ràng buộc giá trị phân biệt

In [14]:
def wrong_allow_duplicates(grid):
    R = len(grid); dp = [-1] * (1 << R); dp[0] = 0
    for r, row in enumerate(grid):                            # SAI: DP theo hàng, không nhớ giá trị đã dùng
        new = dp[:]
        for mask in range(1 << R):
            if dp[mask] >= 0 and not mask >> r & 1:
                new[mask | 1 << r] = max(new[mask | 1 << r], dp[mask] + max(row))
        dp = new
    return max(dp)

print('sai :', wrong_allow_duplicates(examples[0][0]), '| đúng:', Solution().maxScore(examples[0][0]))

sai : 8 | đúng: 8


**Vì sao sai:** DP theo hàng mà không nhớ tập giá trị đã dùng thì không kiểm soát được trùng lặp. Ở ví dụ 1 tình cờ ra đúng `8` (các max theo hàng `3, 4, 1` vốn đã khác nhau), nhưng với `[[5, 1], [5, 2]]` bản sai cộng `5` hai lần:

In [15]:
print('sai :', wrong_allow_duplicates([[5, 1], [5, 2]]), '| đúng:', Solution().maxScore([[5, 1], [5, 2]]))

sai : 10 | đúng: 7


### Lỗi 4 — Cập nhật `dp` tại chỗ trong cùng một giá trị

In [16]:
def wrong_in_place(grid):
    R = len(grid); rows = value_rows(grid); dp = [-1] * (1 << R); dp[0] = 0
    for v, rs in rows.items():
        for mask in range(1 << R):                            # SAI: không dùng bản sao
            if dp[mask] < 0:
                continue
            for r in rs:
                if not mask >> r & 1:
                    dp[mask | 1 << r] = max(dp[mask | 1 << r], dp[mask] + v)
    return max(dp)

print('sai :', wrong_in_place([[7], [7]]), '| đúng:', Solution().maxScore([[7], [7]]))

sai : 14 | đúng: 7


**Vì sao sai:** `dp[01]` vừa được cập nhật bằng `7` lại làm nguồn cho `dp[11] = 14` trong **cùng** vòng của giá trị `7` — dùng `7` hai lần.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hai họ ràng buộc "mỗi hàng ≤ 1", "mỗi giá trị ≤ 1" ⇒ ghép cặp hai phía; chiều nào nhỏ thì làm bitmask.
- Knapsack/DP mà một "món" (giá trị) chỉ dùng một lần ⇒ cập nhật từ bản sao (hoặc duyệt ngược).
- Kích thước 10 ⇒ `2^10 · 100 · 10 = 10^6` là ngân sách hợp lý.

## 15. Biến thể của bài

1. Mỗi hàng chọn **đúng** một ô (bắt buộc) ⇒ chỉ lấy `dp[full]` (có thể không tồn tại).
2. Giới hạn tổng số ô chọn `≤ K`.
3. Ràng buộc theo **cột** thay vì hàng ⇒ chuyển vị.
4. Mỗi giá trị được dùng tối đa `t` lần ⇒ luồng với sức chứa `t` ở cạnh giá trị → đích.

In [17]:
# Biến thể 1 và 4
def exactly_one_per_row(grid):
    R = len(grid); rows = value_rows(grid); dp = [-1] * (1 << R); dp[0] = 0
    for v, rs in rows.items():
        new = dp[:]
        for mask in range(1 << R):
            if dp[mask] >= 0:
                for r in rs:
                    if not mask >> r & 1:
                        new[mask | 1 << r] = max(new[mask | 1 << r], dp[mask] + v)
        dp = new
    return dp[(1 << R) - 1]

def brute_exact(grid):
    best = -1
    def go(r, used, tot):
        nonlocal best
        if r == len(grid):
            best = max(best, tot); return
        for v in set(grid[r]):
            if v not in used:
                used.add(v); go(r + 1, used, tot + v); used.remove(v)
    go(0, set(), 0)
    return best

rng = random.Random(15)
for _ in range(200):
    g = [[rng.randint(1, 5) for _ in range(rng.randint(1, 3))] for _ in range(rng.randint(1, 5))]
    assert exactly_one_per_row(g) == brute_exact(g)
print('Biến thể "mỗi hàng đúng một ô" khớp quay lui (−1 nếu không thể).', exactly_one_per_row([[1], [1]]))

Biến thể "mỗi hàng đúng một ô" khớp quay lui (−1 nếu không thể). -1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Quay lui theo hàng | mũ | Oracle |
| 2 | Đệ quy có nhớ | `O(V 2^R R)` | Dễ viết |
| 3 | DP lặp theo giá trị | `O(V 2^R R)` | **Nộp bài** |
| 4 | Luồng chi phí nhỏ nhất | đa thức | Tổng quát cho `R` lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[2, 2], [2, 2]]`.

**Bài 2 (tính tay):** `[[5, 4], [5, 3], [4, 3]]`.

**Bài 3 (điền TODO):** trả về luôn **các ô** được chọn.

**Bài 4 (tự viết lại):** biến thể 2 — chọn tối đa `K` ô.

**Bài 5 (mini challenge):** biến thể 4 — mỗi giá trị dùng tối đa `t` lần (bằng luồng).

In [18]:
def chosen_cells_todo(grid):
    # TODO
    raise NotImplementedError

def at_most_K_todo(grid, K):
    # TODO
    raise NotImplementedError

def value_capacity_todo(grid, t):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ có giá trị `2`. (2) Dùng một lần. (3) `2`.

**Bài 2.** (1) Ba hàng, ba giá trị `5, 4, 3`. (2) Ghép hoàn hảo: hàng 0–5, hàng 1–3, hàng 2–4. (3) `12`.

**Bài 3.** (1) Lưu `parent[(v, mask)]`. (2) Dựng ngược. (3) Với mỗi cặp `(r, v)` chọn một cột chứa `v`.

**Bài 4.** (1) Thêm chiều số ô — nhưng `popcount(mask)` chính là số ô! (2) Lấy max trên `mask` có `popcount ≤ K`. (3) Không cần thêm trạng thái.

**Bài 5.** (1) Cạnh giá trị → đích sức chứa `t`. (2) Mỗi hàng vẫn tối đa 1, cạnh hàng → giá trị sức chứa 1. (3) Kiểm bằng quay lui đếm số lần dùng.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 4
assert Solution().maxScore([[2, 2], [2, 2]]) == 2
assert Solution().maxScore([[5, 4], [5, 3], [4, 3]]) == 12 == brute([[5, 4], [5, 3], [4, 3]])

def at_most_K(grid, K):
    R = len(grid); rows = value_rows(grid); dp = [-1] * (1 << R); dp[0] = 0
    for v, rs in rows.items():
        new = dp[:]
        for mask in range(1 << R):
            if dp[mask] >= 0:
                for r in rs:
                    if not mask >> r & 1:
                        new[mask | 1 << r] = max(new[mask | 1 << r], dp[mask] + v)
        dp = new
    return max(dp[m] for m in range(1 << R) if bin(m).count('1') <= K)

rng = random.Random(3)
for _ in range(200):
    g = [[rng.randint(1, 8) for _ in range(rng.randint(1, 3))] for _ in range(rng.randint(1, 5))]
    K = rng.randint(1, len(g))
    def go(r, used, tot, k):
        if r == len(g):
            return tot
        res = go(r + 1, used, tot, k)
        if k < K:
            for v in set(g[r]):
                if v not in used:
                    used.add(v); res = max(res, go(r + 1, used, tot + v, k + 1)); used.remove(v)
        return res
    assert at_most_K(g, K) == go(0, set(), 0, 0)
print('Bài 1: 2 | Bài 2: 12 | Bài 4 khớp quay lui.')

Bài 1: 2 | Bài 2: 12 | Bài 4 khớp quay lui.


In [20]:
# Bài 3, 5
def chosen_cells(grid):
    R = len(grid); rows = value_rows(grid)
    vals = list(rows)
    dp = [[-1] * (1 << R) for _ in range(len(vals) + 1)]; dp[0][0] = 0
    for i, v in enumerate(vals):
        for mask in range(1 << R):
            if dp[i][mask] < 0:
                continue
            dp[i + 1][mask] = max(dp[i + 1][mask], dp[i][mask])
            for r in rows[v]:
                if not mask >> r & 1:
                    dp[i + 1][mask | 1 << r] = max(dp[i + 1][mask | 1 << r], dp[i][mask] + v)
    mask = max(range(1 << R), key=lambda m: dp[-1][m])
    cells = []
    for i in range(len(vals), 0, -1):
        v = vals[i - 1]
        if dp[i][mask] == dp[i - 1][mask]:
            continue
        for r in rows[v]:
            if mask >> r & 1 and dp[i - 1][mask ^ 1 << r] >= 0 and dp[i - 1][mask ^ 1 << r] + v == dp[i][mask]:
                cells.append((r, grid[r].index(v))); mask ^= 1 << r
                break
    return sorted(cells)

def value_capacity(grid, t):
    R = len(grid)
    vals = sorted(value_rows(grid)); vid = {v: R + 1 + i for i, v in enumerate(vals)}
    S, T = 0, R + 1 + len(vals); N = T + 1
    graph = [[] for _ in range(N)]
    def add(u, w, cap, cost):
        graph[u].append([w, cap, cost, len(graph[w])]); graph[w].append([u, 0, -cost, len(graph[u]) - 1])
    for r in range(R):
        add(S, r + 1, 1, 0)
        for v in set(grid[r]):
            add(r + 1, vid[v], 1, -v)
    for v in vals:
        add(vid[v], T, t, 0)
    total = 0
    while True:
        dist = [float('inf')] * N; dist[S] = 0; prev = [None] * N
        for _ in range(N):
            ch = False
            for u in range(N):
                if dist[u] < float('inf'):
                    for idx, (w, cap, cost, _) in enumerate(graph[u]):
                        if cap > 0 and dist[u] + cost < dist[w]:
                            dist[w] = dist[u] + cost; prev[w] = (u, idx); ch = True
            if not ch:
                break
        if dist[T] >= 0:
            break
        w = T
        while w != S:
            u, idx = prev[w]; graph[u][idx][1] -= 1; graph[w][graph[u][idx][3]][1] += 1; w = u
        total -= dist[T]
    return total

rng = random.Random(4)
for _ in range(150):
    g = [[rng.randint(1, 6) for _ in range(rng.randint(1, 3))] for _ in range(rng.randint(1, 5))]
    cells = chosen_cells(g)
    assert len({r for r, c in cells}) == len(cells) and len({g[r][c] for r, c in cells}) == len(cells)
    assert sum(g[r][c] for r, c in cells) == Solution().maxScore(g)
    t = rng.randint(1, 3)
    def go(r, cnt):
        if r == len(g):
            return 0
        res = go(r + 1, cnt)
        for v in set(g[r]):
            if cnt.get(v, 0) < t:
                cnt[v] = cnt.get(v, 0) + 1; res = max(res, v + go(r + 1, cnt)); cnt[v] -= 1
        return res
    assert value_capacity(g, t) == go(0, {})
print('Bài 3 (dựng ô) và Bài 5 (sức chứa t) khớp quay lui. Ví dụ:', chosen_cells(examples[0][0]))

Bài 3 (dựng ô) và Bài 5 (sức chứa t) khớp quay lui. Ví dụ: [(0, 2), (1, 0), (2, 0)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
rows[v] = tập hàng chứa v
dp = [-1] * 2^R; dp[0] = 0
for v, rs in rows.items():
    new = dp[:]
    for mask: for r in rs if r ∉ mask: new[mask|1<<r] = max(new[...], dp[mask] + v)
    dp = new
return max(dp)
```

**Checklist:**

- [ ] Duyệt theo **giá trị**, bitmask theo **hàng**?
- [ ] Dùng bản sao `new` (mỗi giá trị một lần)?
- [ ] Không tham lam theo hàng hay theo giá trị?
- [ ] Bỏ qua trạng thái không đạt được (`−1`)?
- [ ] Đáp án là `max` trên mọi `mask`?